# Pipeline A: RadarCVD-Net — Optimized Single-Run Training
### CVPR MSLR 2026 Track 2 — Radar-Only Italian Sign Language Recognition

**Optimized for maximum accuracy within 1.5-day T4×2 budget:**
1. **Single fold** — train on 90% data, validate on 10% (no wasted 5-fold time)
2. **Top-K checkpoint ensemble** — save best 5 checkpoints + SWA model → 6-model ensemble
3. **CutMix + MixUp** — stronger regularization for 126-class problem
4. **EMA (Exponential Moving Average)** — free ensemble member with smoother weights
5. **Enhanced 5-view TTA** — original + time-flip + noise + freq-shift + time-shift
6. **70 epochs** with cosine schedule — longer convergence with single run
7. **Pseudo-CVD + RTM dual-stream** with cross-attention fusion

**Architecture:** EfficientNetV2-S (RTM stream) + ConvNeXt-Tiny (CVD stream) → Cross-Attention Fusion → 126 classes

In [1]:
# ============================================================
# CELL 1: Environment Setup
# ============================================================
import subprocess, sys

def pip_install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg],
                          stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

pip_install('timm')
pip_install('einops')

import torch
print(f'PyTorch {torch.__version__}, CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    n_gpus = torch.cuda.device_count()
    print(f'GPUs available: {n_gpus}')
    for i in range(n_gpus):
        print(f'  GPU {i}: {torch.cuda.get_device_name(i)}')
        mem = torch.cuda.get_device_properties(i).total_memory
        print(f'    VRAM: {mem / 1e9:.1f} GB')
    if n_gpus > 1:
        print(f'\n\U0001F680 Multi-GPU mode enabled: DataParallel across {n_gpus} GPUs')

torch.backends.cudnn.benchmark = True
torch.backends.cudnn.deterministic = False
print('cudnn.benchmark=True -- Tensor Cores enabled')

PyTorch 2.5.1+cu121, CUDA: True
GPUs available: 2
  GPU 0: NVIDIA RTX A6000
    VRAM: 51.0 GB
  GPU 1: NVIDIA RTX A6000
    VRAM: 51.0 GB

🚀 Multi-GPU mode enabled: DataParallel across 2 GPUs
cudnn.benchmark=True -- Tensor Cores enabled


In [2]:
# ============================================================
# CELL 2: Imports & Configuration (Optimized)
# ============================================================
import os, csv, time, math, random, warnings, gc, copy
from pathlib import Path
from collections import defaultdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
from torch.utils.data import Dataset, DataLoader
try:
    from torch.amp import GradScaler, autocast
except ImportError:
    from torch.cuda.amp import GradScaler, autocast
from scipy.signal import windows
from scipy.interpolate import CubicSpline

warnings.filterwarnings('ignore')

MODE = 'compete'

class CFG:
    seed           = 42
    device         = 'cuda' if torch.cuda.is_available() else 'cpu'
    num_gpus       = torch.cuda.device_count() if torch.cuda.is_available() else 1
    num_classes    = 126
    num_workers    = 0 if MODE == 'debug' else 4

    # Data
    img_size       = 224
    max_time       = 48
    n_fft          = 128
    expected_submission_rows = 9828
    strict_submission_rows = True

    # Runtime
    skip_existing_checkpoints = True

    # Backbones
    rtm_backbone   = 'tf_efficientnetv2_s.in21k_ft_in1k'
    cvd_backbone   = 'convnext_tiny.fb_in22k_ft_in1k'
    feat_dim       = 512

    # Training — OPTIMIZED for single-run with checkpoint ensemble
    val_frac       = 0.1        # 10% holdout for validation
    epochs         = 70 if MODE == 'compete' else 3
    lr             = 3e-4       # slightly higher LR with longer warmup
    batch_size     = (24 * num_gpus) if MODE == 'compete' else 8  # conservative for T4
    weight_decay   = 0.05
    label_smoothing = 0.1
    warmup_epochs  = 5          # longer warmup
    grad_clip      = 1.0
    use_amp        = True

    # Augmentation — enhanced
    mixup_alpha    = 0.4
    cutmix_alpha   = 1.0        # NEW: CutMix
    mix_prob       = 0.5        # 50% chance of MixUp or CutMix
    noise_std      = 0.02
    freq_mask      = 30
    time_mask      = 10         # slightly more aggressive

    # SWA
    swa_frac       = 0.80       # start SWA at 80% of training
    swa_lr         = 1e-5

    # EMA
    ema_decay      = 0.9995     # NEW: exponential moving average

    # Ensemble
    top_k_save     = 5          # NEW: save top-5 checkpoints for ensemble

    # TTA
    use_tta        = True
    tta_views      = 5          # NEW: 5-view TTA


def set_seed(s):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.benchmark = True

set_seed(CFG.seed)
print(f'MODE={MODE}, device={CFG.device}, GPUs={CFG.num_gpus}')
print(f'Batch size: {CFG.batch_size} (scaled for {CFG.num_gpus} GPU(s))')
print(f'Epochs: {CFG.epochs}, LR: {CFG.lr}, Val fraction: {CFG.val_frac}')
print(f'Top-K checkpoints: {CFG.top_k_save}, TTA views: {CFG.tta_views}')
print(f'EMA decay: {CFG.ema_decay}')
print(f'RTM backbone: {CFG.rtm_backbone}')
print(f'CVD backbone: {CFG.cvd_backbone}')

MODE=compete, device=cuda, GPUs=2
Batch size: 48 (scaled for 2 GPU(s))
Epochs: 70, LR: 0.0003, Val fraction: 0.1
Top-K checkpoints: 5, TTA views: 5
EMA decay: 0.9995
RTM backbone: tf_efficientnetv2_s.in21k_ft_in1k
CVD backbone: convnext_tiny.fb_in22k_ft_in1k


In [3]:
# ============================================================
# CELL 3: Auto-detect data paths
# ============================================================
def find_data():
    for p in [
        Path('/kaggle/input/competitions/cvpr-mslr-2026-track-2'),
        Path('/kaggle/input/2st-multimodal-italian-sign-language-rec'),
        Path('/kaggle/input/cvpr-mslr-2026-track-2'),
        Path('d:/Current-Research/CVPR2026-SignEval/cvpr-mslr-2026-track-2'),
        Path('../cvpr-mslr-2026-track-2 (1)'),
        # Path('../cvpr-mslr-2026-track-2'),
    ]:
        if (p / 'train').exists():
            return p
    raise FileNotFoundError('Competition dataset not found! Attach it in Kaggle Input.')

DATA_ROOT = find_data()
TRAIN_DIR = DATA_ROOT / 'train'
VAL_DIR   = DATA_ROOT / 'val'
TEST_DIR  = DATA_ROOT / 'test'
OUT       = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('./output')
OUT.mkdir(parents=True, exist_ok=True)

print(f'Data root: {DATA_ROOT}')
print(f'Train dir: {TRAIN_DIR}')
print(f'Val dir  : {VAL_DIR}')
print(f'Test dir : {TEST_DIR} (exists={TEST_DIR.exists()})')
print(f'Output   : {OUT}')

Data root: ../cvpr-mslr-2026-track-2 (1)
Train dir: ../cvpr-mslr-2026-track-2 (1)/train
Val dir  : ../cvpr-mslr-2026-track-2 (1)/val
Test dir : ../cvpr-mslr-2026-track-2 (1)/test (exists=True)
Output   : output


In [4]:
# ============================================================
# CELL 4: Index samples + single stratified train/val split
# ============================================================
def build_train_samples(train_dir):
    samples = []
    class_dirs = sorted(
        [d for d in Path(train_dir).iterdir()
         if d.is_dir() and d.name.split('_')[0].isdigit()],
        key=lambda d: int(d.name.split('_')[0]))
    class_names = [d.name for d in class_dirs]
    for cd in class_dirs:
        label = int(cd.name.split('_')[0])
        for sd in sorted(cd.iterdir()):
            if sd.is_dir() and sd.name.startswith('SAMPLE_'):
                if (sd / f'{sd.name}_RTM1.npy').exists():
                    samples.append((str(sd), label))
    return samples, class_names

def stratified_split(samples, val_frac=0.1, seed=42):
    """Single stratified train/val split — more training data than K-fold."""
    rng = np.random.RandomState(seed)
    c2i = defaultdict(list)
    for i, (_, l) in enumerate(samples):
        c2i[l].append(i)
    train_idx, val_idx = [], []
    for cls in sorted(c2i):
        idxs = c2i[cls].copy()
        rng.shuffle(idxs)
        n_val = max(1, int(len(idxs) * val_frac))
        val_idx.extend(idxs[:n_val])
        train_idx.extend(idxs[n_val:])
    return train_idx, val_idx

def _sample_id_sort_key(sample_id):
    token = str(sample_id).replace('SAMPLE_', '')
    return int(token) if token.isdigit() else str(sample_id)

def build_test_samples(sample_dir):
    sample_dir = Path(sample_dir)
    found = []
    for sd in sorted(sample_dir.rglob('SAMPLE_*')):
        if not sd.is_dir():
            continue
        sid = sd.name
        if not sid.startswith('SAMPLE_'):
            continue
        if (sd / f'{sid}_RTM1.npy').exists():
            found.append((str(sd), sid))
    by_sid = {}
    for sample_dir_path, sid in found:
        if sid not in by_sid:
            by_sid[sid] = sample_dir_path
    samples = [(by_sid[sid], sid) for sid in sorted(by_sid, key=_sample_id_sort_key)]
    return samples

def build_all_test_samples():
    all_samples = []
    seen_sids = set()
    for split_dir in [VAL_DIR, TEST_DIR]:
        if not split_dir.exists():
            print(f'  Warning: {split_dir} does not exist, skipping')
            continue
        split_samples = build_test_samples(split_dir)
        added = 0
        for sample_dir, sid in split_samples:
            if sid not in seen_sids:
                seen_sids.add(sid)
                all_samples.append((sample_dir, sid))
                added += 1
        print(f'  {split_dir.name}/: {added} unique samples')
    all_samples.sort(key=lambda x: _sample_id_sort_key(x[1]))
    return all_samples

all_train_samples, class_names = build_train_samples(TRAIN_DIR)
train_idx, val_idx = stratified_split(all_train_samples, CFG.val_frac, CFG.seed)

print(f'Scanning val/ and test/ for submission samples...')
test_samples = build_all_test_samples()

print(f'\nTotal labeled samples: {len(all_train_samples)}, {len(class_names)} classes')
print(f'Train split: {len(train_idx)} samples ({100*(1-CFG.val_frac):.0f}%)')
print(f'Val split:   {len(val_idx)} samples ({100*CFG.val_frac:.0f}%)')
print(f'Test:        {len(test_samples)} samples (val + test dirs)')

if CFG.strict_submission_rows and MODE == 'compete':
    if len(test_samples) != CFG.expected_submission_rows:
        raise RuntimeError(
            f'Expected {CFG.expected_submission_rows} test samples, found {len(test_samples)}. '
            'Check dataset mount/path completeness.'
        )
    print(f'\u2705 Test sample count matches expected: {CFG.expected_submission_rows}')

# Sanity check
sd0, lbl0 = all_train_samples[0]
sid0 = Path(sd0).name
arr0 = np.load(os.path.join(sd0, f'{sid0}_RTM1.npy'))
print(f'\nSample: {sid0}, label={lbl0}, RTM1 shape={arr0.shape}, range=[{arr0.min():.1f}, {arr0.max():.1f}]')

Scanning val/ and test/ for submission samples...
  val/: 4914 unique samples
  test/: 4914 unique samples

Total labeled samples: 14742, 126 classes
Train split: 13356 samples (90%)
Val split:   1386 samples (10%)
Test:        9828 samples (val + test dirs)
✅ Test sample count matches expected: 9828

Sample: SAMPLE_10462, label=0, RTM1 shape=(16, 256), range=[-108.8, -21.8]


In [5]:
# ============================================================
# CELL 5: Pseudo-CVD Computation
# ============================================================
def compute_cvd(rtm_db, n_fft=128):
    T, R = rtm_db.shape
    rtm_linear = np.power(10.0, rtm_db / 20.0)
    win = windows.blackmanharris(T).astype(np.float32)
    rtm_windowed = rtm_linear * win[:, np.newaxis]
    if T < n_fft:
        rtm_padded = np.pad(rtm_windowed, ((0, n_fft - T), (0, 0)), mode='constant')
    else:
        rtm_padded = rtm_windowed[:n_fft, :]
    cvd_complex = np.fft.rfft(rtm_padded, axis=0)
    cvd_mag = np.abs(cvd_complex[1:, :])
    cvd_db = 20.0 * np.log10(cvd_mag + 1e-10)
    return cvd_db.T

# Sanity check
sd0, _ = all_train_samples[0]
rtm_test = np.load(os.path.join(sd0, f'{Path(sd0).name}_RTM1.npy')).astype(np.float32)
cvd_test = compute_cvd(rtm_test, n_fft=CFG.n_fft)
print(f'RTM shape: {rtm_test.shape} -> CVD shape: {cvd_test.shape}')
print(f'CVD computation OK')

RTM shape: (16, 256) -> CVD shape: (256, 64)
CVD computation OK


In [6]:
# ============================================================
# CELL 6: Physics-Aware Radar Augmentation (enhanced with CutMix)
# ============================================================
class RadarPhysicsAug:
    @staticmethod
    def time_warp(rtm, sigma=0.15):
        T, R = rtm.shape
        if T < 4:
            return rtm
        orig_indices = np.arange(T, dtype=np.float32)
        perturbations = np.random.normal(0, sigma, T).cumsum()
        warp_indices = orig_indices + perturbations
        for i in range(1, T):
            if warp_indices[i] <= warp_indices[i-1]:
                warp_indices[i] = warp_indices[i-1] + 0.01
        warp_indices = (warp_indices - warp_indices[0]) / (warp_indices[-1] - warp_indices[0]) * (T - 1)
        warped = np.zeros_like(rtm)
        for r in range(R):
            cs = CubicSpline(orig_indices, rtm[:, r])
            warped[:, r] = cs(warp_indices)
        return warped

    @staticmethod
    def magnitude_warp(rtm, sigma=0.1, n_knots=4):
        T, R = rtm.shape
        if T < 2:
            return rtm
        n_knots = min(n_knots, T)
        knot_x = np.linspace(0, T-1, n_knots)
        knot_y = 1.0 + np.random.normal(0, sigma, n_knots)
        cs = CubicSpline(knot_x, knot_y)
        curve = cs(np.arange(T))
        return rtm * curve[:, np.newaxis]

    @staticmethod
    def simulated_multipath(rtm, max_delay=10, atten_range=(0.05, 0.15)):
        T, R = rtm.shape
        delay = np.random.randint(3, max_delay + 1)
        atten = np.random.uniform(*atten_range)
        ghost = np.zeros_like(rtm)
        if delay < R:
            ghost[:, delay:] = rtm[:, :R - delay] * atten
        return rtm + ghost

    @staticmethod
    def antenna_dropout(rtms_stacked, p=0.1):
        if np.random.random() < p:
            drop = np.random.randint(0, 3)
            rtms_stacked[drop] = np.zeros_like(rtms_stacked[drop])
        return rtms_stacked

print('Physics-aware augmentation module ready')


def rand_bbox(H, W, lam):
    """Generate random bounding box for CutMix."""
    cut_rat = np.sqrt(1.0 - lam)
    cut_h = int(H * cut_rat)
    cut_w = int(W * cut_rat)
    cy = np.random.randint(H)
    cx = np.random.randint(W)
    y1 = np.clip(cy - cut_h // 2, 0, H)
    y2 = np.clip(cy + cut_h // 2, 0, H)
    x1 = np.clip(cx - cut_w // 2, 0, W)
    x2 = np.clip(cx + cut_w // 2, 0, W)
    return y1, y2, x1, x2

print('CutMix utility ready')

Physics-aware augmentation module ready
CutMix utility ready


In [7]:
# ============================================================
# CELL 7: Dual-Stream Dataset (RTM + CVD) — with in-memory caching
# ============================================================
class DualStreamRTMDataset(Dataset):
    def __init__(self, sample_list, img_size=224, max_T=48, n_fft=128, augment=False):
        self.samples = sample_list
        self.img_size = img_size
        self.max_T = max_T
        self.n_fft = n_fft
        self.augment = augment
        self.aug = RadarPhysicsAug()

        print(f'  Pre-loading {len(sample_list)} samples into RAM...', end=' ', flush=True)
        self._cache = []
        for sd, _ in sample_list:
            sid = Path(sd).name
            rtms = []
            for i in range(1, 4):
                arr = np.load(os.path.join(sd, f'{sid}_RTM{i}.npy')).astype(np.float32)
                rtms.append(arr)
            self._cache.append(rtms)
        print('done.', flush=True)

    def __len__(self):
        return len(self.samples)

    def _load_raw(self, idx):
        return [rtm.copy() for rtm in self._cache[idx]]

    def _augment_raw(self, rtms):
        augmented = []
        for rtm in rtms:
            if random.random() < 0.3:
                rtm = self.aug.time_warp(rtm, sigma=0.15)
            if random.random() < 0.3:
                rtm = self.aug.magnitude_warp(rtm, sigma=0.1)
            if random.random() < 0.15:
                rtm = self.aug.simulated_multipath(rtm)
            if random.random() < 0.3:
                rtm = rtm[::-1, :].copy()
            augmented.append(rtm)
        stacked = np.stack(augmented, axis=0)
        stacked = self.aug.antenna_dropout(stacked)
        return [stacked[i] for i in range(3)]

    def _make_rtm_image(self, rtms_db):
        processed = []
        for rtm in rtms_db:
            T, R = rtm.shape
            if T >= self.max_T:
                s = (T - self.max_T) // 2
                rtm = rtm[s:s + self.max_T, :]
            else:
                pad = self.max_T - T
                pl, pr = pad // 2, pad - pad // 2
                rtm = np.pad(rtm, ((pl, pr), (0, 0)), mode='constant')
            processed.append(rtm)
        stacked = np.stack(processed, axis=0)
        stacked = stacked.transpose(0, 2, 1)
        mn, mx = stacked.min(), stacked.max()
        stacked = (stacked - mn) / (mx - mn + 1e-8)
        tensor = torch.from_numpy(stacked).float()
        tensor = F.interpolate(
            tensor.unsqueeze(0), size=(self.img_size, self.img_size),
            mode='bilinear', align_corners=False
        ).squeeze(0)
        return tensor

    def _make_cvd_image(self, rtms_db):
        cvds = []
        for rtm in rtms_db:
            cvd = compute_cvd(rtm, n_fft=self.n_fft)
            cvds.append(cvd)
        stacked = np.stack(cvds, axis=0)
        mn, mx = stacked.min(), stacked.max()
        stacked = (stacked - mn) / (mx - mn + 1e-8)
        tensor = torch.from_numpy(stacked).float()
        tensor = F.interpolate(
            tensor.unsqueeze(0), size=(self.img_size, self.img_size),
            mode='bilinear', align_corners=False
        ).squeeze(0)
        return tensor

    def _spec_augment(self, img):
        C, H, W = img.shape
        # Frequency masking (up to 2 masks)
        for _ in range(random.randint(1, 2)):
            if random.random() < 0.5:
                f = random.randint(1, CFG.freq_mask)
                f0 = random.randint(0, max(0, H - f))
                img[:, f0:f0+f, :] = 0
        # Time masking (up to 2 masks)
        for _ in range(random.randint(1, 2)):
            if random.random() < 0.5:
                t = random.randint(1, CFG.time_mask)
                t0 = random.randint(0, max(0, W - t))
                img[:, :, t0:t0+t] = 0
        return img

    def __getitem__(self, idx):
        _, label_or_id = self.samples[idx]
        rtms_db = self._load_raw(idx)

        if self.augment:
            rtms_db = self._augment_raw(rtms_db)

        rtm_img = self._make_rtm_image(rtms_db)
        cvd_img = self._make_cvd_image(rtms_db)

        if self.augment:
            rtm_img = self._spec_augment(rtm_img)
            cvd_img = self._spec_augment(cvd_img)

        return rtm_img, cvd_img, label_or_id


# Sanity check
_ds = DualStreamRTMDataset(all_train_samples[:4], img_size=CFG.img_size,
                            max_T=CFG.max_time, n_fft=CFG.n_fft, augment=True)
rtm0, cvd0, lbl0 = _ds[0]
print(f'RTM image: {rtm0.shape}, range=[{rtm0.min():.3f}, {rtm0.max():.3f}]')
print(f'CVD image: {cvd0.shape}, range=[{cvd0.min():.3f}, {cvd0.max():.3f}]')
print(f'Label: {lbl0}')
assert rtm0.shape == (3, 224, 224)
assert cvd0.shape == (3, 224, 224)
print('Dual-stream dataset sanity check PASSED')
del _ds

  Pre-loading 4 samples into RAM... done.
RTM image: torch.Size([3, 224, 224]), range=[0.000, 1.000]
CVD image: torch.Size([3, 224, 224]), range=[0.000, 1.000]
Label: 0
Dual-stream dataset sanity check PASSED


In [8]:
# ============================================================
# CELL 8: Cross-Antenna Self-Attention Module
# ============================================================
class CrossAntennaSelfAttention(nn.Module):
    def __init__(self, in_channels=3, reduction=1, num_heads=1):
        super().__init__()
        self.feat_extract = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
        )
        self.attn = nn.MultiheadAttention(16, num_heads=num_heads, batch_first=True)
        self.norm = nn.LayerNorm(16)
        self.channel_gate = nn.Sequential(
            nn.Linear(16, 8),
            nn.ReLU(inplace=True),
            nn.Linear(8, 1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        B, C, H, W = x.shape
        feats = []
        for i in range(C):
            f = self.feat_extract(x[:, i:i+1, :, :])
            feats.append(f)
        tokens = torch.stack(feats, dim=1)
        attn_out, _ = self.attn(tokens, tokens, tokens)
        attn_out = self.norm(attn_out + tokens)
        weights = []
        for i in range(C):
            w = self.channel_gate(attn_out[:, i, :])
            weights.append(w)
        weights = torch.stack(weights, dim=1).unsqueeze(-1)
        return x * weights

# Test
_casa = CrossAntennaSelfAttention().to(CFG.device)
_x = torch.randn(2, 3, 224, 224).to(CFG.device)
with torch.no_grad():
    _out = _casa(_x)
print(f'CASA: {_x.shape} -> {_out.shape}')
print(f'CASA params: {sum(p.numel() for p in _casa.parameters()) / 1e3:.1f}K')
del _casa, _x, _out; torch.cuda.empty_cache()

CASA: torch.Size([2, 3, 224, 224]) -> torch.Size([2, 3, 224, 224])
CASA params: 1.5K


In [9]:
# ============================================================
# CELL 9: Asymmetric Cross-Attention Fusion
# ============================================================
class AsymmetricCrossAttentionFusion(nn.Module):
    def __init__(self, feat_dim, num_heads=8, dropout=0.1):
        super().__init__()
        self.cross_attn = nn.MultiheadAttention(
            feat_dim, num_heads, dropout=dropout, batch_first=True
        )
        self.norm1 = nn.LayerNorm(feat_dim)
        self.norm2 = nn.LayerNorm(feat_dim)
        self.ffn = nn.Sequential(
            nn.Linear(feat_dim, feat_dim * 4),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(feat_dim * 4, feat_dim),
            nn.Dropout(dropout),
        )
        self.gate = nn.Sequential(
            nn.Linear(feat_dim * 2, feat_dim),
            nn.Sigmoid()
        )

    def forward(self, rtm_feat, cvd_feat):
        q = rtm_feat.unsqueeze(1)
        kv = cvd_feat.unsqueeze(1)
        attn_out, _ = self.cross_attn(q, kv, kv)
        attn_out = self.norm1(attn_out + q).squeeze(1)
        ffn_out = self.ffn(attn_out)
        enhanced = self.norm2(ffn_out + attn_out)
        g = self.gate(torch.cat([rtm_feat, enhanced], dim=1))
        fused = g * enhanced + (1 - g) * rtm_feat
        return fused

# Test
_fuse = AsymmetricCrossAttentionFusion(512).to(CFG.device)
_a = torch.randn(2, 512).to(CFG.device)
_b = torch.randn(2, 512).to(CFG.device)
with torch.no_grad():
    _out = _fuse(_a, _b)
print(f'Fusion: ({_a.shape}, {_b.shape}) -> {_out.shape}')
del _fuse, _a, _b, _out; torch.cuda.empty_cache()

Fusion: (torch.Size([2, 512]), torch.Size([2, 512])) -> torch.Size([2, 512])


In [10]:
# ============================================================
# CELL 10: RadarCVD-Net Full Model
# ============================================================
class RadarCVDNet(nn.Module):
    def __init__(self, num_classes=126, feat_dim=512,
                 rtm_name='tf_efficientnetv2_s.in21k_ft_in1k',
                 cvd_name='convnext_tiny.fb_in22k_ft_in1k',
                 pretrained=True):
        super().__init__()
        self.casa_rtm = CrossAntennaSelfAttention()
        self.casa_cvd = CrossAntennaSelfAttention()

        self.rtm_backbone = timm.create_model(
            rtm_name, pretrained=pretrained, num_classes=0,
            in_chans=3, drop_rate=0.3, drop_path_rate=0.2,
        )
        rtm_feat_dim = self.rtm_backbone.num_features

        self.cvd_backbone = timm.create_model(
            cvd_name, pretrained=pretrained, num_classes=0,
            in_chans=3, drop_rate=0.3, drop_path_rate=0.2,
        )
        cvd_feat_dim = self.cvd_backbone.num_features

        if hasattr(self.rtm_backbone, 'set_grad_checkpointing'):
            self.rtm_backbone.set_grad_checkpointing(True)
        if hasattr(self.cvd_backbone, 'set_grad_checkpointing'):
            self.cvd_backbone.set_grad_checkpointing(True)

        self.rtm_proj = nn.Linear(rtm_feat_dim, feat_dim)
        self.cvd_proj = nn.Linear(cvd_feat_dim, feat_dim)
        self.fusion = AsymmetricCrossAttentionFusion(feat_dim, num_heads=8)

        self.head_rtm = nn.Linear(feat_dim, num_classes)
        self.head_cvd = nn.Linear(feat_dim, num_classes)

        self.head = nn.Sequential(
            nn.LayerNorm(feat_dim),
            nn.Dropout(0.3),
            nn.Linear(feat_dim, num_classes),
        )

        n_params = sum(p.numel() for p in self.parameters()) / 1e6
        print(f'RadarCVD-Net: {n_params:.1f}M params (RTM: {rtm_name}, CVD: {cvd_name})')

    def forward(self, rtm_img, cvd_img, return_aux=False):
        rtm_attended = self.casa_rtm(rtm_img)
        cvd_attended = self.casa_cvd(cvd_img)
        rtm_feat = self.rtm_proj(self.rtm_backbone(rtm_attended))
        cvd_feat = self.cvd_proj(self.cvd_backbone(cvd_attended))
        fused = self.fusion(rtm_feat, cvd_feat)
        logits = self.head(fused)
        if return_aux:
            return logits, self.head_rtm(rtm_feat), self.head_cvd(cvd_feat)
        return logits

# Forward pass test
_model = RadarCVDNet(
    num_classes=CFG.num_classes, feat_dim=CFG.feat_dim,
    rtm_name=CFG.rtm_backbone, cvd_name=CFG.cvd_backbone,
).to(CFG.device)
_r = torch.randn(2, 3, 224, 224).to(CFG.device)
_c = torch.randn(2, 3, 224, 224).to(CFG.device)
with torch.no_grad():
    _logits, _aux_r, _aux_c = _model(_r, _c, return_aux=True)
print(f'Forward test: main={_logits.shape}, aux_rtm={_aux_r.shape}, aux_cvd={_aux_c.shape}')
assert _logits.shape == (2, 126)
print('Model forward pass PASSED')
del _model, _r, _c, _logits, _aux_r, _aux_c
gc.collect(); torch.cuda.empty_cache()

RadarCVD-Net: 52.9M params (RTM: tf_efficientnetv2_s.in21k_ft_in1k, CVD: convnext_tiny.fb_in22k_ft_in1k)
Forward test: main=torch.Size([2, 126]), aux_rtm=torch.Size([2, 126]), aux_cvd=torch.Size([2, 126])
Model forward pass PASSED


In [11]:
# ============================================================
# CELL 11: Training Utilities (MixUp, CutMix, EMA, LR schedule)
# ============================================================
def cosine_lr(optimizer, warmup, total, min_frac=0.01):
    def fn(ep):
        if ep < warmup:
            return (ep + 1) / warmup
        prog = (ep - warmup) / max(1, total - warmup)
        return min_frac + 0.5 * (1 - min_frac) * (1 + math.cos(math.pi * prog))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, fn)


def mixup_dual(rtm, cvd, labels, alpha=0.4):
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    idx = torch.randperm(rtm.size(0), device=rtm.device)
    rtm_mix = lam * rtm + (1 - lam) * rtm[idx]
    cvd_mix = lam * cvd + (1 - lam) * cvd[idx]
    return rtm_mix, cvd_mix, labels, labels[idx], lam


def cutmix_dual(rtm, cvd, labels, alpha=1.0):
    """CutMix: cut and paste patches between samples (both streams)."""
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    idx = torch.randperm(rtm.size(0), device=rtm.device)
    _, _, H, W = rtm.shape
    y1, y2, x1, x2 = rand_bbox(H, W, lam)

    rtm_mix = rtm.clone()
    cvd_mix = cvd.clone()
    rtm_mix[:, :, y1:y2, x1:x2] = rtm[idx, :, y1:y2, x1:x2]
    cvd_mix[:, :, y1:y2, x1:x2] = cvd[idx, :, y1:y2, x1:x2]

    # Adjust lambda to actual area ratio
    lam = 1 - ((y2 - y1) * (x2 - x1)) / (H * W)
    return rtm_mix, cvd_mix, labels, labels[idx], lam


class ModelEMA:
    """Exponential Moving Average of model weights — free ensemble member."""
    def __init__(self, model, decay=0.9995):
        self.ema = copy.deepcopy(model)
        self.ema.eval()
        self.decay = decay
        for p in self.ema.parameters():
            p.requires_grad_(False)

    @torch.no_grad()
    def update(self, model):
        for ema_p, model_p in zip(self.ema.parameters(), model.parameters()):
            ema_p.data.mul_(self.decay).add_(model_p.data, alpha=1.0 - self.decay)
        # Also update buffers (BN running stats)
        for ema_b, model_b in zip(self.ema.buffers(), model.buffers()):
            ema_b.data.copy_(model_b.data)

    def state_dict(self):
        return self.ema.state_dict()


@torch.no_grad()
def validate_dual(model, loader):
    model.eval()
    correct = total = 0
    for rtm_imgs, cvd_imgs, labels in loader:
        rtm_imgs = rtm_imgs.to(CFG.device, non_blocking=True)
        cvd_imgs = cvd_imgs.to(CFG.device, non_blocking=True)
        labels = labels.to(CFG.device, non_blocking=True)
        with autocast('cuda', enabled=CFG.use_amp):
            logits = model(rtm_imgs, cvd_imgs)
        correct += (logits.argmax(1) == labels).sum().item()
        total += labels.size(0)
    return 100.0 * correct / max(total, 1)

print('Training utilities ready (MixUp + CutMix + EMA + cosine LR)')

Training utilities ready (MixUp + CutMix + EMA + cosine LR)


In [12]:
# ============================================================
# CELL 12: Single-Run Training with Top-K Checkpoint Ensemble
# ============================================================
import heapq

TOTAL_START = time.time()
swa_start_ep = int(CFG.epochs * CFG.swa_frac)
AUX_WEIGHT = 0.3
LOG_INTERVAL = 10

sep = "=" * 70
print(f"\n{sep}")
print(f"RadarCVD-Net | epochs={CFG.epochs} | lr={CFG.lr} | bs={CFG.batch_size}")
print(f"SWA@{swa_start_ep} | EMA decay={CFG.ema_decay} | Top-K={CFG.top_k_save}")
print(f"Aux loss weight: {AUX_WEIGHT}")
print(f"MixUp alpha={CFG.mixup_alpha}, CutMix alpha={CFG.cutmix_alpha}, prob={CFG.mix_prob}")
print(sep)

# Build datasets
train_samples = [all_train_samples[i] for i in train_idx]
val_samples = [all_train_samples[i] for i in val_idx]
print(f"Train: {len(train_samples)}, Val: {len(val_samples)}")

train_ds = DualStreamRTMDataset(
    train_samples, img_size=CFG.img_size, max_T=CFG.max_time,
    n_fft=CFG.n_fft, augment=True
)
val_ds = DualStreamRTMDataset(
    val_samples, img_size=CFG.img_size, max_T=CFG.max_time,
    n_fft=CFG.n_fft, augment=False
)

train_loader = DataLoader(train_ds, batch_size=CFG.batch_size, shuffle=True,
                          num_workers=CFG.num_workers, pin_memory=True, drop_last=True,
                          persistent_workers=True if CFG.num_workers > 0 else False)
val_loader = DataLoader(val_ds, batch_size=CFG.batch_size * 2, shuffle=False,
                        num_workers=CFG.num_workers, pin_memory=True,
                        persistent_workers=True if CFG.num_workers > 0 else False)

n_batches = len(train_loader)
print(f"Batches/epoch: {n_batches}")

# Build model
model = RadarCVDNet(
    num_classes=CFG.num_classes, feat_dim=CFG.feat_dim,
    rtm_name=CFG.rtm_backbone, cvd_name=CFG.cvd_backbone,
).to(CFG.device)

# Multi-GPU
raw_model = model  # keep reference for EMA
if CFG.num_gpus > 1:
    model = nn.DataParallel(model)
    print(f"DataParallel enabled across {CFG.num_gpus} GPUs")

opt = torch.optim.AdamW(model.parameters(), lr=CFG.lr, weight_decay=CFG.weight_decay)
sched = cosine_lr(opt, CFG.warmup_epochs, CFG.epochs)
scaler = GradScaler('cuda', enabled=CFG.use_amp)
crit = nn.CrossEntropyLoss(label_smoothing=CFG.label_smoothing)

# SWA
swa_model = torch.optim.swa_utils.AveragedModel(model)
swa_sched = torch.optim.swa_utils.SWALR(opt, swa_lr=CFG.swa_lr)

# EMA
ema = ModelEMA(raw_model, decay=CFG.ema_decay)

# Top-K checkpoint tracking: min-heap of (val_acc, epoch, path)
top_k_heap = []
best_acc = 0.0

for ep in range(CFG.epochs):
    t0 = time.time()
    model.train()
    loss_sum = 0.0
    correct = total = 0

    for batch_idx, (rtm_imgs, cvd_imgs, labels) in enumerate(train_loader):
        rtm_imgs = rtm_imgs.to(CFG.device, non_blocking=True)
        cvd_imgs = cvd_imgs.to(CFG.device, non_blocking=True)
        labels = labels.to(CFG.device, non_blocking=True)

        # MixUp or CutMix with equal probability
        do_mix = random.random() < CFG.mix_prob and ep < CFG.epochs - 3
        if do_mix:
            if random.random() < 0.5:
                rtm_imgs, cvd_imgs, ya, yb, lam = mixup_dual(
                    rtm_imgs, cvd_imgs, labels, CFG.mixup_alpha)
            else:
                rtm_imgs, cvd_imgs, ya, yb, lam = cutmix_dual(
                    rtm_imgs, cvd_imgs, labels, CFG.cutmix_alpha)

        with autocast('cuda', enabled=CFG.use_amp):
            logits, aux_rtm, aux_cvd = model(rtm_imgs, cvd_imgs, return_aux=True)

            if do_mix:
                loss_main = lam * crit(logits, ya) + (1 - lam) * crit(logits, yb)
                loss_rtm = lam * crit(aux_rtm, ya) + (1 - lam) * crit(aux_rtm, yb)
                loss_cvd = lam * crit(aux_cvd, ya) + (1 - lam) * crit(aux_cvd, yb)
            else:
                loss_main = crit(logits, labels)
                loss_rtm = crit(aux_rtm, labels)
                loss_cvd = crit(aux_cvd, labels)

            loss = loss_main + AUX_WEIGHT * (loss_rtm + loss_cvd)

        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        nn.utils.clip_grad_norm_(model.parameters(), CFG.grad_clip)
        scaler.step(opt)
        scaler.update()
        opt.zero_grad(set_to_none=True)

        # Update EMA
        ema.update(raw_model)

        loss_sum += loss.item()
        if not do_mix:
            correct += (logits.argmax(1) == labels).sum().item()
            total += labels.size(0)

        if (batch_idx + 1) % LOG_INTERVAL == 0 or (batch_idx + 1) == n_batches:
            elapsed = time.time() - t0
            avg_loss = loss_sum / (batch_idx + 1)
            acc_so_far = 100.0 * correct / max(total, 1)
            print(f"    Ep {ep} [{batch_idx+1}/{n_batches}] "
                  f"loss={avg_loss:.3f} acc={acc_so_far:.1f}% "
                  f"({elapsed:.0f}s)", flush=True)

    sched.step()
    if ep >= swa_start_ep:
        swa_model.update_parameters(model)
        swa_sched.step()

    val_acc = validate_dual(model, val_loader)
    dt = time.time() - t0
    ep_train_acc = 100.0 * correct / max(total, 1)
    avg_loss = loss_sum / max(n_batches, 1)
    print(f"  Ep {ep:3d} | loss={avg_loss:.3f} | train={ep_train_acc:.1f}% | "
          f"val={val_acc:.1f}% | lr={opt.param_groups[0]['lr']:.2e} | {dt:.0f}s")

    # Save Top-K checkpoints
    ckpt_path = OUT / f"ckpt_ep{ep}.pt"
    model_state = model.module.state_dict() if CFG.num_gpus > 1 else model.state_dict()
    if len(top_k_heap) < CFG.top_k_save:
        torch.save({'model': model_state, 'acc': val_acc, 'epoch': ep}, ckpt_path)
        heapq.heappush(top_k_heap, (val_acc, ep, str(ckpt_path)))
        print(f"    >>> Saved checkpoint (top-{len(top_k_heap)}): {val_acc:.2f}%")
    elif val_acc > top_k_heap[0][0]:
        # Remove worst checkpoint
        worst_acc, worst_ep, worst_path = heapq.heapreplace(
            top_k_heap, (val_acc, ep, str(ckpt_path)))
        if os.path.exists(worst_path):
            os.remove(worst_path)
        torch.save({'model': model_state, 'acc': val_acc, 'epoch': ep}, ckpt_path)
        print(f"    >>> Saved checkpoint (replaced {worst_acc:.2f}%): {val_acc:.2f}%")

    if val_acc > best_acc:
        best_acc = val_acc
        print(f"    >>> New best: {best_acc:.2f}%")

# Save EMA model
ema_path = OUT / "ema_model.pt"
torch.save({'model': ema.state_dict(), 'acc': best_acc, 'epoch': CFG.epochs, 'type': 'ema'}, ema_path)
print(f"\nEMA model saved: {ema_path}")

# SWA finalize
try:
    swa_model.train()
    momenta = {}
    for name, module in swa_model.named_modules():
        if isinstance(module, (nn.BatchNorm1d, nn.BatchNorm2d, nn.BatchNorm3d)):
            module.reset_running_stats()
            momenta[module] = module.momentum
            module.momentum = None
    with torch.no_grad():
        for rtm_imgs, cvd_imgs, _ in train_loader:
            rtm_imgs = rtm_imgs.to(CFG.device, non_blocking=True)
            cvd_imgs = cvd_imgs.to(CFG.device, non_blocking=True)
            swa_model(rtm_imgs, cvd_imgs)
    for module in momenta:
        module.momentum = momenta[module]

    swa_acc = validate_dual(swa_model, val_loader)
    swa_path = OUT / "swa_model.pt"
    if CFG.num_gpus > 1:
        swa_state = swa_model.module.module.state_dict()
    else:
        swa_state = swa_model.module.state_dict()
    torch.save({'model': swa_state, 'acc': swa_acc, 'epoch': CFG.epochs, 'type': 'swa'}, swa_path)
    print(f"SWA val acc: {swa_acc:.2f}% (saved: {swa_path})")
except Exception as e:
    print(f"SWA BN update failed: {e}")
    swa_path = None

total_time = time.time() - TOTAL_START
print(f"\n{sep}")
print(f"Training complete in {total_time/3600:.2f}h")
print(f"Best val acc: {best_acc:.2f}%")
print(f"Top-K checkpoints: {sorted([(a, e) for a, e, p in top_k_heap], reverse=True)}")
print(sep)

# Build list of all model paths for ensemble
all_model_paths = [p for _, _, p in sorted(top_k_heap, reverse=True)]
all_model_paths.append(str(ema_path))
if swa_path and swa_path.exists():
    all_model_paths.append(str(swa_path))
print(f"Total ensemble models: {len(all_model_paths)}")

del model, swa_model, opt, sched, scaler
gc.collect(); torch.cuda.empty_cache()


RadarCVD-Net | epochs=70 | lr=0.0003 | bs=48
SWA@56 | EMA decay=0.9995 | Top-K=5
Aux loss weight: 0.3
MixUp alpha=0.4, CutMix alpha=1.0, prob=0.5
Train: 13356, Val: 1386
  Pre-loading 13356 samples into RAM... done.
  Pre-loading 1386 samples into RAM... done.
Batches/epoch: 278
RadarCVD-Net: 52.9M params (RTM: tf_efficientnetv2_s.in21k_ft_in1k, CVD: convnext_tiny.fb_in22k_ft_in1k)
DataParallel enabled across 2 GPUs
    Ep 0 [10/278] loss=7.948 acc=1.4% (97s)
    Ep 0 [20/278] loss=7.983 acc=0.8% (165s)
    Ep 0 [30/278] loss=7.966 acc=0.8% (233s)
    Ep 0 [40/278] loss=7.975 acc=0.8% (300s)
    Ep 0 [50/278] loss=7.971 acc=0.8% (369s)
    Ep 0 [60/278] loss=7.969 acc=0.8% (436s)
    Ep 0 [70/278] loss=7.964 acc=0.8% (504s)
    Ep 0 [80/278] loss=7.966 acc=1.0% (571s)
    Ep 0 [90/278] loss=7.966 acc=0.9% (641s)
    Ep 0 [100/278] loss=7.960 acc=0.9% (707s)
    Ep 0 [110/278] loss=7.956 acc=0.9% (775s)
    Ep 0 [120/278] loss=7.951 acc=0.9% (843s)
    Ep 0 [130/278] loss=7.944 acc=0.8

In [13]:
# ============================================================
# CELL 13: Ensemble Inference with Enhanced 5-View TTA
# ============================================================
@torch.no_grad()
def predict_tta_dual(model, loader, n_views=5):
    """Enhanced 5-view TTA: original + time-reverse + noise + freq-shift + time-shift."""
    model.eval()
    probs_list, ids_list = [], []

    for rtm_imgs, cvd_imgs, sids in loader:
        rtm_imgs = rtm_imgs.to(CFG.device, non_blocking=True)
        cvd_imgs = cvd_imgs.to(CFG.device, non_blocking=True)

        with autocast('cuda', enabled=CFG.use_amp):
            p = F.softmax(model(rtm_imgs, cvd_imgs), dim=1)

        if n_views >= 2:
            # TTA 1: time-reverse both streams
            with autocast('cuda', enabled=CFG.use_amp):
                p1 = F.softmax(model(
                    torch.flip(rtm_imgs, [3]),
                    torch.flip(cvd_imgs, [3])
                ), dim=1)
            p = p + p1

        if n_views >= 3:
            # TTA 2: slight Gaussian noise
            with autocast('cuda', enabled=CFG.use_amp):
                p2 = F.softmax(model(
                    rtm_imgs + torch.randn_like(rtm_imgs) * 0.01,
                    cvd_imgs + torch.randn_like(cvd_imgs) * 0.01
                ), dim=1)
            p = p + p2

        if n_views >= 4:
            # TTA 3: frequency shift (roll along H axis)
            shift = 3
            with autocast('cuda', enabled=CFG.use_amp):
                p3 = F.softmax(model(
                    torch.roll(rtm_imgs, shifts=shift, dims=2),
                    torch.roll(cvd_imgs, shifts=shift, dims=2)
                ), dim=1)
            p = p + p3

        if n_views >= 5:
            # TTA 4: small time shift (roll along W axis)
            shift = 2
            with autocast('cuda', enabled=CFG.use_amp):
                p4 = F.softmax(model(
                    torch.roll(rtm_imgs, shifts=shift, dims=3),
                    torch.roll(cvd_imgs, shifts=shift, dims=3)
                ), dim=1)
            p = p + p4

        p = p / min(n_views, 5)
        probs_list.append(p.cpu())
        ids_list.extend(sids if isinstance(sids[0], str) else sids.tolist())

    return torch.cat(probs_list), ids_list


if len(all_model_paths) == 0:
    raise RuntimeError('No checkpoints available. Run Cell 12 first.')

print(f'Generating predictions from {len(all_model_paths)} models ({CFG.tta_views}-view TTA)...')

test_ds = DualStreamRTMDataset(
    test_samples, img_size=CFG.img_size, max_T=CFG.max_time,
    n_fft=CFG.n_fft, augment=False
)
test_loader = DataLoader(test_ds, batch_size=CFG.batch_size * 2, shuffle=False,
                          num_workers=CFG.num_workers, pin_memory=True,
                          persistent_workers=True if CFG.num_workers > 0 else False)

ensemble_probs = None
sample_ids = None

for i, path in enumerate(all_model_paths):
    ckpt = torch.load(path, map_location='cpu', weights_only=False)
    acc = ckpt.get('acc', 0)
    ep = ckpt.get('epoch', '?')
    mtype = ckpt.get('type', 'checkpoint')
    print(f'  [{i+1}/{len(all_model_paths)}] type={mtype} epoch={ep} val={acc:.1f}%')

    model = RadarCVDNet(
        num_classes=CFG.num_classes, feat_dim=CFG.feat_dim,
        rtm_name=CFG.rtm_backbone, cvd_name=CFG.cvd_backbone,
        pretrained=False
    ).to(CFG.device)
    model.load_state_dict(ckpt['model'])

    probs, ids = predict_tta_dual(model, test_loader, n_views=CFG.tta_views)

    if ensemble_probs is None:
        ensemble_probs = probs
        sample_ids = ids
    else:
        ensemble_probs += probs

    del model, ckpt
    gc.collect(); torch.cuda.empty_cache()

ensemble_probs /= len(all_model_paths)
preds = ensemble_probs.argmax(dim=1).numpy()

if CFG.strict_submission_rows and MODE == 'compete':
    if len(preds) != CFG.expected_submission_rows:
        raise ValueError(
            f'Prediction count mismatch: expected {CFG.expected_submission_rows}, got {len(preds)}'
        )

print(f'\nPredictions: {len(preds)} samples')
print(f'Unique classes predicted: {len(np.unique(preds))} / {CFG.num_classes}')

Generating predictions from 7 models (5-view TTA)...
  Pre-loading 9828 samples into RAM... done.
  [1/7] type=checkpoint epoch=60 val=84.6%
RadarCVD-Net: 52.9M params (RTM: tf_efficientnetv2_s.in21k_ft_in1k, CVD: convnext_tiny.fb_in22k_ft_in1k)
  [2/7] type=checkpoint epoch=68 val=84.5%
RadarCVD-Net: 52.9M params (RTM: tf_efficientnetv2_s.in21k_ft_in1k, CVD: convnext_tiny.fb_in22k_ft_in1k)
  [3/7] type=checkpoint epoch=57 val=84.5%
RadarCVD-Net: 52.9M params (RTM: tf_efficientnetv2_s.in21k_ft_in1k, CVD: convnext_tiny.fb_in22k_ft_in1k)
  [4/7] type=checkpoint epoch=56 val=84.5%
RadarCVD-Net: 52.9M params (RTM: tf_efficientnetv2_s.in21k_ft_in1k, CVD: convnext_tiny.fb_in22k_ft_in1k)
  [5/7] type=checkpoint epoch=66 val=84.4%
RadarCVD-Net: 52.9M params (RTM: tf_efficientnetv2_s.in21k_ft_in1k, CVD: convnext_tiny.fb_in22k_ft_in1k)
  [6/7] type=ema epoch=70 val=84.6%
RadarCVD-Net: 52.9M params (RTM: tf_efficientnetv2_s.in21k_ft_in1k, CVD: convnext_tiny.fb_in22k_ft_in1k)
  [7/7] type=swa epoc

In [14]:
# ============================================================
# CELL 14: Write Submission CSV
# ============================================================
rows = []
for sid, pred in zip(sample_ids, preds):
    nid = int(sid.replace('SAMPLE_', '')) if isinstance(sid, str) else int(sid)
    rows.append({'id': nid, 'Pred': int(pred)})

rows.sort(key=lambda r: r['id'])

if CFG.strict_submission_rows:
    if len(rows) != CFG.expected_submission_rows:
        raise ValueError(
            f'Submission row count mismatch: expected {CFG.expected_submission_rows}, got {len(rows)}'
        )
    unique_ids = len({r['id'] for r in rows})
    if unique_ids != CFG.expected_submission_rows:
        raise ValueError(
            f'Submission id uniqueness mismatch: expected {CFG.expected_submission_rows}, got {unique_ids}'
        )
    print(f'\u2705 Row count + unique IDs verified: {CFG.expected_submission_rows}')

sub_path = OUT / 'submission.csv'
with open(sub_path, 'w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=['id', 'Pred'])
    w.writeheader()
    w.writerows(rows)

print(f'Submission saved: {sub_path}')
print(f'Total rows: {len(rows)}')

# Sanity checks
import pandas as pd
df = pd.read_csv(sub_path)
print(f'\n--- Sanity Check ---')
print(f'Shape:       {df.shape}')
print(f'Pred range:  [{df.Pred.min()}, {df.Pred.max()}]')
print(f'Unique Pred: {df.Pred.nunique()}')
print(f'Any NaN:     {df.isnull().any().any()}')
print(f'\nHead:')
print(df.head(10))
print(f'\nClass distribution (top 10):')
print(df.Pred.value_counts().head(10))

✅ Row count + unique IDs verified: 9828
Submission saved: output/submission.csv
Total rows: 9828

--- Sanity Check ---
Shape:       (9828, 2)
Pred range:  [0, 125]
Unique Pred: 126
Any NaN:     False

Head:
   id  Pred
0   0   121
1   1    23
2   4    84
3   6   101
4   7    81
5  10    40
6  12    39
7  13    52
8  17    88
9  20    24

Class distribution (top 10):
Pred
56     106
46     102
51     100
97      97
31      94
89      93
108     92
52      89
2       89
100     87
Name: count, dtype: int64


In [15]:
# ============================================================
# CELL 15: Save ensemble probabilities for cross-pipeline fusion
# ============================================================
np.save(OUT / 'pipeline_a_probs.npy', ensemble_probs.numpy())
with open(OUT / 'pipeline_a_ids.txt', 'w') as f:
    for sid in sample_ids:
        f.write(f'{sid}\n')

print(f'Pipeline A probabilities saved for cross-pipeline ensemble')
print(f'Shape: {ensemble_probs.shape}')

Pipeline A probabilities saved for cross-pipeline ensemble
Shape: torch.Size([9828, 126])


---
## Pipeline A: RadarCVD-Net — Optimized Summary

### Key Optimizations vs Original
1. **Single fold** — trains on 90% of data (vs 80% per fold), saves 4× time
2. **Top-5 checkpoint ensemble + EMA + SWA** — 7 models from single training run
3. **CutMix + MixUp** — stronger regularization, especially for 126-class problem
4. **EMA (decay=0.9995)** — free ensemble member with smoother weights
5. **5-view TTA** — original + time-flip + noise + freq-shift + time-shift
6. **70 epochs** with 5-epoch warmup — longer convergence on more data
7. **Higher LR (3e-4)** with longer cosine schedule

### Expected Performance
- Single run + ensemble: **82-88%** (vs ~80% with original 5-fold)
- Time budget: **~18-24h** on T4×2 (fits 1.5 day budget)